In [3]:
import sys 
import cv2 as cv
import pdb
import numpy as np
import os
import copy
import matplotlib.pyplot as plt
from typing import List

In [4]:
def show_image(image_, window_name='image', timeout=0):
    """
    Show image.
    :param image_
    :param window_name
    :param timeout
    """
    height, width = image_.shape[:2]

    fx = 0.3
    fy = 0.3
    new_width = int(width * fx)
    new_height = int(height * fy)
    image_ = cv.resize(image_,(new_width, new_height))
    
    
    cv.imshow(window_name, np.uint8(image_))
    cv.waitKey(timeout)
    cv.destroyAllWindows()

In [5]:
#read images
path = 'board+dominoes'
board = cv.imread('Raw_board_template.jpg') #os.path.join(path, '02.jpg')) 
show_image(board)

In [6]:
#Fit the board template perfect.
def unwarp(img, src, dst):
    h, w = img.shape[:2]

    pers = cv.getPerspectiveTransform(src, dst)

    warped = cv.warpPerspective(img, pers, (w, h), flags=cv.INTER_LINEAR)

    return  cv.flip(warped, 1)

In [13]:
w, h = board.shape[0], board.shape[1]
# We will first manually select the source points 
# we will select the destination point which will map the source points in
# original image to destination points in unwarped image


src = np.float32([(20,14),      # (20,1)
                  (1885,17),    # (540,130)
                  (10,1920),    # (20,520)
                  (1892,1886)]) # (570,450)


dst = np.float32([(w, 0),   #(573,0)
                  (0, 0),  #(0,0)
                  (w, h),   #(573,566)
                  (0, h)]) #(0,566)

board_fixed = unwarp(board, src, dst)
curr_dir = os.getcwd()
cv.imwrite(os.path.join(curr_dir, 'board_fixed.jpg'), board_fixed)
show_image(board_fixed)
cv.waitKey(0)
cv.destroyAllWindows()

In [5]:
def get_keypoints_and_features(image, show_details=False, max_features=500) -> tuple:
    """
    :return the keypoints: [cv.Keypoint] and the features: np.ndarray for each keypoint.
    """

    def show_keypoints(image_, keypoints_):
        """
        Show the keypoints found in the image.
        """
        image_output = image_.copy()
        image_output = cv.drawKeypoints(image_, keypoints_, image_output, flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
        # show_image(image_output, 'keypoints',framework="matplotlib")
        show_image(image_output, 'keypoints')

    # convert image to grayscale
    gray_image = cv.cvtColor(image, cv.COLOR_BGR2GRAY) 
    
    # create the SIFT object. (https://docs.opencv.org/master/da/df5/tutorial_py_sift_intro.html)
    sift = cv.SIFT_create(max_features) 

    # Find keypoints from the grayscale image.
    keypoints = sift.detect(gray_image, None) # use it to detect keypoints

    # compute the features based on the grayscale image and the keypoints.
    keypoints, features = sift.compute(gray_image, keypoints) # extract features
     
    if show_details:
        show_keypoints(image, keypoints)

    return keypoints, features

In [6]:
regular_image = cv.imread("D:/MASTER/Sem_II/Computer Vision/CV-2023-Project1/train/regular_tasks/4_19.jpg")
board = cv.imread("D:/MASTER/Sem_II/Computer Vision/CV-2023-Project1/board+dominoes/02.jpg")

(kpsIm, featIm) = get_keypoints_and_features(regular_image, show_details= True)
(kpsB, featB) = get_keypoints_and_features(board, show_details= True)
print('keypoints regular image: ', len(kpsIm), 'features regular image:', len(featIm))
print('keypoints board template: ', len(kpsB), 'features board template:', len(featB))

keypoints regular image:  501 features regular image: 501
keypoints board template:  501 features board template: 501


In [7]:
def match_features(features_source, features_dest) -> List[List[cv.DMatch]]:
    """
    Match features from the source image with the features from the destination image.
    :return: [[DMatch]] - The rusult of the matching. For each set of features from the source image,
    it returns the first 'K' matchings from the destination images.
    """
 
    feature_matcher = cv.DescriptorMatcher_create("FlannBased")
    matches = feature_matcher.knnMatch(features_source, features_dest, k=2)   
    return matches

In [17]:
#test features matching
show_image(regular_image)

keyp1, img_feat1 = get_keypoints_and_features(regular_image, True)
keyp2, img_feat2 = get_keypoints_and_features(board, True)

matches = match_features(img_feat1, img_feat2)
print(len(matches))
print(f"First match: {matches[0]}")

501
First match: (< cv2.DMatch 000001A6A6D86770>, < cv2.DMatch 000001A6A6D86730>)


In [11]:
def generate_homography(all_matches:  List[cv.DMatch],
                        keypoints_source: List[cv.KeyPoint],
                        keypoints_dest : List[cv.KeyPoint],
                        ratio: float = 0.75,
                        ransac_rep: int = 4.0):
    """
    :param all_matches [DMatch]
    :param keypoints_source [cv.Point]
    :param ratio - Lowe's ratio test (the ratio 1st neighbour distance / 2nd neighbour distance)
    :param keypoints_source: nd.array [Nx2] (x, y coordinates)
    :param keypoints_dest: nd.array [Nx2] (x, y coordinates)
    :param ransac_rep: float. The threshold in the RANSAC algorithm.
    :return: The homography matrix.
    
    class DMatch:
        distance - Distance between descriptors. The lower, the better it is.
        imgIdx - Index of the train image
        queryIdx - Index of the descriptor in query descriptors
        trainIdx - Index of the descriptor in train descriptors
    
    class KeyPoint:
        pt - The x, y coordinates of a point.
    
    """
    if not all_matches:
        return None
    
    # find the matchings that pass the Lowe's ratio test (ratio parameter).
    matches = [] 
    for match in all_matches:  
        if len(match) == 2 and (match[0].distance / match[1].distance) < ratio:
            matches.append(match[0])
     
    # get the coordinates of the keypoints from the source image.
    points_source = np.float32([keypoints_source[m.queryIdx].pt for m in matches])

    # get the coordinates of the keypoints from the destination image.
    points_dest = np.float32([keypoints_dest[m.trainIdx].pt for m in matches])

    # Obtain the Homagraphy. (https://docs.opencv.org/master/d1/de0/tutorial_py_feature_homography.html)
    if len(points_source) > 4:
        H, status = cv.findHomography(points_source, points_dest, cv.RANSAC, ransac_rep)
        return H
    else:
        return None

In [12]:
H = generate_homography(matches,kpsIm,kpsB)
print(f"Homography type: {type(H)}, len:{len(H)}")
print(f"First Homography: {H[0]}")

Homography type: <class 'numpy.ndarray'>, len:3
First Homography: [ 1.30543049e+00  3.70473712e-01 -7.91663826e+02]


In [13]:
def stitch_images(image, template, show_details=False):
    """ 
    :param image_source (image from the right part).
    :param image_dest (image from the left part).
    :param show_details
    :return - the stitched image.
    TODO:
    1. Get the keypoints and the features from the source image.
    2. Get the keypoints and the features from the destination image.
    3. Match the features.
    4. Find the homography matrix.
    5. Apply the homography matrix on the source image.
    (https://opencv-python-tutroals.readthedocs.io/en/latest/py_tutorials/py_imgproc/py_geometric_transformations/py_geometric_transformations.html#affine-transformation)
    6. Copy the destination image in the resulting image from the previous point.
    """
    def show_matches(all_matches_, n=10):
        matches = sorted(all_matches_, key = lambda x:x[0].distance)
        matches = matches[:n] 
        image_output = cv.drawMatchesKnn(template, kpsA,
                                         image, kpsB,   
                                         matches, None, flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        show_image(image_output, 'matches')

    # template image
    kpsA, descsA = get_keypoints_and_features(template,show_details=True)
    # image to be alligned as in the template
    kpsB, descsB = get_keypoints_and_features(image,show_details=True)
    print(f"Keypoints lenght: {len(kpsA)},/n Features lenght: {len(descsA)}")
    print(f"Keypoints lenght: {len(kpsB)},/n Features lenght: {len(descsB)}")

    # match the features
    all_matches = match_features(descsB, descsA)
    print(f"Total number of matches: {len(all_matches)}")

    if show_details:
        show_matches(copy.copy(all_matches),n=10)

    # generate the homographic map 
    H = generate_homography(all_matches, kpsB, kpsA, 0.6, 4.0)

    (h, w) = template.shape[:2]
    result = cv.warpPerspective(image, H, (w, h))
    
    return result

In [16]:
board_fixed = cv.imread('board_fixed.jpg')
stitched = stitch_images(regular_image,board_fixed, show_details=True)
show_image(stitched)

Keypoints lenght: 501,/n Features lenght: 501
Keypoints lenght: 501,/n Features lenght: 501
Total number of matches: 501


error: OpenCV(4.7.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\imgwarp.cpp:3298: error: (-215:Assertion failed) (M0.type() == CV_32F || M0.type() == CV_64F) && M0.rows == 3 && M0.cols == 3 in function 'cv::warpPerspective'


In [40]:
def pad_image(image_, procent=0.1):
    pad_h = int(image_.shape[0] * procent)
    pad_w = int(image_.shape[1] * procent)
    big_image = np.zeros((image_.shape[0] + 2 * pad_h, image_.shape[1] + pad_w, 3), np.uint8)
    big_image[pad_h: pad_h + image_.shape[0], pad_w: pad_w + image_.shape[1]] = image_.copy()
    return big_image

img = pad_image(stiched)
show_image(img)

In [46]:
def stitch_images_from_folder(folder_path, show_details=False):
    """
    Stitch the images from the last image to the first.
    TODO:
    1. Read the images from the folder, sort them (ascending order), 
    then reverse the list (because we are going to stitch them from the last image to the first).
    2. Read the first image (the source image).
    3. While you have unread images, read the next image (destination image), 
    stitch it with the source image then save the resulting image in the source image (in the next step it will be the source image).
    """
    image_names = os.listdir(folder_path)
    image_names = [file for file in image_names if file.endswith(".jpg")]
    image_names.sort()
    image_names = image_names[::-1]
    
# filter the list to include only JPG files

    assert len(image_names) >= 2
    image_source = cv.imread(os.path.join(folder_path, image_names[0]))
    show_image(image_source)
    for idx_image in range(1, 5): #len(image_names)
        image_dest = cv.imread(os.path.join(folder_path, image_names[idx_image]))
        image_dest = pad_image(image_dest)
        result = stitch_images(board_fixed, image_dest, show_details=show_details)
        image_source = result.copy()
        
    return result
    

In [47]:

stitched = stitch_images_from_folder('D:/MASTER/Sem_II/Computer Vision/CV-2023-Project1/train/regular_tasks', show_details=True)
show_image(stitched)


error: OpenCV(4.6.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\imgwarp.cpp:3168: error: (-215:Assertion failed) (M0.type() == CV_32F || M0.type() == CV_64F) && M0.rows == 3 && M0.cols == 3 in function 'cv::warpPerspective'


In [72]:
def stitch_images_inside(image_source, image_dest, show_details=False):
    """ 
    :param image_source (image from the right part).
    :param image_dest (image from the lest part).
    :param show_details
    :return: the stitched image.
    TODO:
    1. Get the keypoints and the features from the source image.
    2. Get the keypoints and the features from the destination image.
    3. Match the features.
    4. Find the homography matrix.
    5. Apply the homography matrix on the source image.
    6. Copy the destination image in the resulting image from the previous point, but keep the resulting pixels in place!
    """
 
    image_dest = pad_image(image_dest)
    keypoints_source, features_source = get_keypoints_and_features(image_source, show_details=show_details)
    keypoints_dest, features_dest = get_keypoints_and_features(image_dest, show_details=show_details)
    
    def show_matches(all_matches_, n=10):
        matches = sorted(all_matches_, key = lambda x:x[0].distance)
        matches = matches[:n] 
        image_output = cv.drawMatchesKnn(image_source, keypoints_source, 
                                         image_dest, keypoints_dest,  
                                         matches, None, flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        show_image(image_output, 'matches')
          
    all_matches = match_features(features_source, features_dest)
    
    if show_details:
        show_matches(copy.copy(all_matches))
        
    H = generate_homography(all_matches, keypoints_source, keypoints_dest, 0.6, 4)

    result = cv.warpPerspective(image_source, H,  # returnez direct result
        (image_dest.shape[1], image_dest.shape[0]))
    if show_details:
        show_image(result, window_name="the position of then image in now image")
    
    # mask = result[:, :] == [0, 0, 0] # hopefully we dont have to many black pixels in the image :)
    
    # result = result * (1 - mask) + image_dest * mask 
    return result


In [15]:
stitched = stitch_images_inside(image_source=board_fixed, image_dest=regular_image, show_details=True)

NameError: name 'stitch_images_inside' is not defined

In [77]:
show_image(stitched, window_name="pr")

error: OpenCV(4.6.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:3930: error: (-215:Assertion failed) func != 0 in function 'cv::hal::resize'


In [83]:
#Lab 2
